# Phase M2-B1 — Unified Embedding Evaluation Battery (18 Tests)
## CNN (nnUNet) + ViT (SwinUNETR) on MU-Glioma-Post

**Same test battery as BraTS Phase 2/3** — ensures consistent evaluation across datasets:
- **M1-M6**: Morphology tests (volume R², log-vol R², enhancement, necrosis F1, core fraction, patient purity)
- **H1-H5**: Heterogeneity tests (RankMe, diversity, responder F1, norm CV, RankMe standalone)
- **T1-T8**: Temporal tests (Spearman, near-duplicate, delta R², RANO AUC, coherence, velocity, Cohen's d, Kendall τ)

### Kaggle Datasets Required:
1. `mu-glioma-m1-outputs` — M1 pipeline output (`scan_index.json`, `data_splits.json`, `tumor_volumes.csv`, `mu_glioma_master.csv`)
2. `mu-glioma-m2-nnunet` — M2_A1 output (`cnn_nnunet_embeddings.npz`)
3. *(Optional)* `mu-glioma-m2-swinunetr` — M2_A2 output (`vit_swinunetr_embeddings.npz`)

> **Incremental mode**: This notebook works with **CNN-only** or **both models**. Run with nnUNet results first — it saves `m2_eval_results.json`. Re-run later with SwinUNETR added to evaluate + compare both.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CACHE MODE: if cached results JSON found, skip M1-T8 and
# jump straight to dashboard + plots
# ═══════════════════════════════════════════════════════════
SKIP_IF_CACHED = True

import warnings, json as _json, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import r2_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr, kendalltau as kt
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_ROOT = Path("/kaggle/working/phase_m2_evaluation")
FIG_DIR = OUTPUT_ROOT / "figures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_npz(patterns):
    for pat in patterns:
        for root in SEARCH_ROOTS:
            matches = list(root.rglob(f"{pat}.npz"))
            if matches: return matches[0]
    return None

def find_file(patterns):
    for pat in patterns:
        for root in SEARCH_ROOTS:
            matches = list(root.rglob(pat))
            if matches: return matches[0]
    return None

def load_embedding(path, model_key):
    data = np.load(path, allow_pickle=True)
    embs_arr = data["embeddings"]
    pids_arr = data["patient_ids"]
    tps_arr  = data["timepoints"]
    emb_dict = {}
    for i in range(len(embs_arr)):
        key = f"{pids_arr[i]}__{tps_arr[i]}"
        emb_dict[key] = embs_arr[i]
    return emb_dict, embs_arr.shape[1]

def match_row(tumor_df, pid, tp):
    if tumor_df is None: return None
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp))]
    if len(m) > 0: return m.iloc[0]
    # Try numeric fallback
    try:
        tp_int = int(tp)
        m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                     (tumor_df["timepoint"].astype(int) == tp_int)]
        if len(m) > 0: return m.iloc[0]
    except (ValueError, TypeError):
        pass
    return None

# ── Try loading cached results ──
cached_json = find_file(["m2_eval_results.json"])
CACHE_LOADED = False

# -- Check for NEW embeddings not yet cached --
cnn_path_check = find_npz(["cnn_nnunet_embeddings", "nnunet_embeddings"])
vit_path_check = find_npz(["vit_swinunetr_embeddings", "swinunetr_embeddings"])

if SKIP_IF_CACHED and cached_json is not None:
    with open(cached_json) as f:
        cached_results = _json.load(f)
    cached_models = set(cached_results.keys())
    
    # Check if new models are available that aren't cached yet
    new_models_available = []
    if vit_path_check and "swinunetr" not in cached_models:
        new_models_available.append("swinunetr")
    if cnn_path_check and "nnunet" not in cached_models:
        new_models_available.append("nnunet")
    
    if not new_models_available:
        # Pure cache mode - all models already evaluated
        results = cached_results
        out_path = OUTPUT_ROOT / "m2_eval_results.json"
        if str(cached_json) != str(out_path):
            with open(out_path, "w") as f:
                _json.dump(results, f, indent=2)
        models = {k: {} for k in results}
        CACHE_LOADED = True
        print(f"CACHE LOADED: {cached_json}")
        print(f"   Models: {list(results.keys())}")
        # Also load ablation caches if present
        abl_json = find_file(["m2_ablation_results.json"])
        if abl_json:
            with open(abl_json) as f:
                results_ablation = _json.load(f)
            print(f"   Ablation (no-vol) cache: {abl_json}")
        else:
            results_ablation = {}
        seg_json = find_file(["m2_segment_ablation.json"])
        if seg_json:
            with open(seg_json) as f:
                results_seg = _json.load(f)
            print(f"   Segment ablation cache: {seg_json}")
        else:
            results_seg = {}
        print(f"   Skipping tests M1-T8. Running plots + dashboard only.")
    else:
        # New model(s) found - run tests for new, keep cached for old
        results = cached_results  # keep existing
        CACHE_LOADED = False
        print(f"Cached results for: {list(cached_models)}")
        print(f"   NEW model(s) detected: {new_models_available}")
        print(f"   Running tests for new model(s), keeping cached results.")
if not CACHE_LOADED:
    # ── Load embeddings ──
    if "models" not in dir() or not isinstance(models, dict):
        models = {}
    
    # CNN (nnUNet)
    cnn_path = find_npz(["cnn_nnunet_embeddings", "nnunet_embeddings"])
    if cnn_path:
        embs_cnn, dim_cnn = load_embedding(cnn_path, "nnunet")
        models["nnunet"] = embs_cnn
        print(f"nnunet: {len(embs_cnn)} scans | dim={dim_cnn} | File: {cnn_path}")
    else:
        print("WARNING: No CNN embeddings found")
    
    # ViT (SwinUNETR)
    vit_path = find_npz(["vit_swinunetr_embeddings", "swinunetr_embeddings"])
    if vit_path:
        embs_vit, dim_vit = load_embedding(vit_path, "swinunetr")
        models["swinunetr"] = embs_vit
        print(f"swinunetr: {len(embs_vit)} scans | dim={dim_vit} | File: {vit_path}")
    else:
        print("WARNING: No ViT embeddings found")
    
    if not models:
        raise FileNotFoundError(
            "No embeddings found. Attach cnn_nnunet_embeddings.npz or vit_swinunetr_embeddings.npz")

    # ── RAW EMBEDDING STATISTICS ──
    print("\n" + "="*60)
    print("  RAW EMBEDDING STATISTICS")
    print("="*60)
    for mn, emb_dict in models.items():
        keys = sorted(emb_dict.keys())
        X_raw = np.stack([emb_dict[k] for k in keys])
        norms = np.linalg.norm(X_raw, axis=1)
        pids = set(k.split("__")[0] for k in keys)
        D = X_raw.shape[1]
        print(f"\n  {mn}: {len(keys)} scans | {len(pids)} patients | dim={D}")
        print(f"    Norm: min={norms.min():.1f}  max={norms.max():.1f}  mean={norms.mean():.1f}")
        print(f"    Norm CV (std/mean): {norms.std()/norms.mean():.4f}")
        if D >= 2121:
            C = (D - 9) // 11
            print(f"    Architecture: C={C} | octant={8*C}D  region={3*C}D  vol=9D")
        idx = np.random.choice(len(keys), size=min(1000, len(keys)), replace=False)
        pairs = [(idx[i], idx[i+1]) for i in range(0, len(idx)-1, 2)]
        cos_sims = []
        for a, b in pairs[:500]:
            na, nb_ = np.linalg.norm(X_raw[a]), np.linalg.norm(X_raw[b])
            if na > 1e-8 and nb_ > 1e-8:
                cos_sims.append(np.dot(X_raw[a], X_raw[b]) / (na * nb_))
        cos_sims = np.array(cos_sims)
        print(f"    Cosine sim ({len(cos_sims)} pairs): mean={cos_sims.mean():.3f}  std={cos_sims.std():.3f}")
        X_unit = X_raw / (norms[:, None] + 1e-8)
        subset = X_unit[np.random.choice(len(X_unit), min(500, len(X_unit)), replace=False)]
        dists = [np.linalg.norm(subset[i] - subset[j])
                 for i in range(len(subset)) for j in range(i+1, min(i+20, len(subset)))]
        print(f"    Diversity (mean pairwise L2): {np.mean(dists):.3f}")
        svs = np.linalg.svd(X_unit[:min(500, len(X_unit))], compute_uv=False)
        p = svs / svs.sum(); p = p[p > 1e-10]
        print(f"    RankMe (effective rank): {float(np.exp(-np.sum(p * np.log(p)))):.1f}")
        print(f"    Zero embeddings (resected): {np.sum(norms < 1e-5)}")

    # ── COMPONENT-WISE L2 NORMALISATION ──
    print("\n" + "="*60)
    print("  APPLYING COMPONENT-WISE L2 NORMALISATION")
    print("="*60)
    for mn in list(models.keys()):
        emb_dict = models[mn]
        keys = list(emb_dict.keys())
        arr = np.stack([emb_dict[k] for k in keys])
        D = arr.shape[1]
        if D >= 2121:
            C = (D - 9) // 11
            oct_d, reg_d = 8 * C, 3 * C
            comp_o = arr[:, :oct_d] / (np.linalg.norm(arr[:, :oct_d], axis=1, keepdims=True) + 1e-8)
            comp_r = arr[:, oct_d:oct_d+reg_d] / (np.linalg.norm(arr[:, oct_d:oct_d+reg_d], axis=1, keepdims=True) + 1e-8)
            comp_v = arr[:, oct_d+reg_d:] / (np.linalg.norm(arr[:, oct_d+reg_d:], axis=1, keepdims=True) + 1e-8)
            arr_n = np.concatenate([comp_o, comp_r, comp_v * 2], axis=1)
            print(f"  {mn}: Component-wise L2 (D={D}, C={C})")
        else:
            arr_n = arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-8)
            print(f"  {mn}: Global L2 (D={D})")
        models[mn] = {k: arr_n[i] for i, k in enumerate(keys)}
        norms_n = np.linalg.norm(arr_n, axis=1)
        idx = np.random.choice(len(keys), min(1000, len(keys)), replace=False)
        pairs = [(idx[i], idx[i+1]) for i in range(0, len(idx)-1, 2)]
        cos_p = [np.dot(arr_n[a], arr_n[b]) / (np.linalg.norm(arr_n[a])*np.linalg.norm(arr_n[b]) + 1e-8)
                 for a, b in pairs[:500]
                 if np.linalg.norm(arr_n[a]) > 1e-8 and np.linalg.norm(arr_n[b]) > 1e-8]
        print(f"    Norm: mean={norms_n.mean():.3f}  CV={norms_n.std()/norms_n.mean():.4f}")
        print(f"    Cosine sim ({len(cos_p)} pairs): mean={np.mean(cos_p):.3f}")

    # ── LOAD TUMOUR VOLUME METADATA ──
    tumor_df = None
    for root in SEARCH_ROOTS:
        for f in root.rglob("tumor_volumes.csv"):
            tumor_df = pd.read_csv(f)
            print(f"\nTumor volumes: {len(tumor_df)} rows from {f}")
            break
        if tumor_df is not None: break
    if tumor_df is not None:
        keys = list(list(models.values())[0].keys())
        pid0, tp0 = keys[0].split("__")
        r = match_row(tumor_df, pid0, tp0)
        print(f"Match test: {'FOUND' if r is not None else 'NOT FOUND'}")

    results = {}

print(f"\n✅ Ready | CACHE_LOADED={CACHE_LOADED} | Models: {list(models.keys())}")
print(f"Output: {OUTPUT_ROOT}")

In [ ]:
if not CACHE_LOADED:
    # M1-M6: MORPHOLOGY TESTS
    print("=" * 60)
    print("  MORPHOLOGY TESTS M1-M6")
    print("=" * 60)
    
    for mn, embs in models.items():
        keys = list(embs.keys())
        X    = np.stack([embs[k] for k in keys])
        X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        Xs   = StandardScaler().fit_transform(X_l2)
        if mn not in results: results[mn] = {}
    
        mX, mvols = [], {"wt": [], "tc": [], "et": []}
        for k in keys:
            pid, tp = k.split("__")
            row = match_row(tumor_df, pid, tp)
            if row is not None:
                mX.append(Xs[keys.index(k)])
                for r_name in ["wt", "tc", "et"]:
                    for col in [f"{r_name}_vol", f"{r_name.upper()}_vol",
                                 f"{r_name}_volume", f"vol_{r_name}",
                                 f"{r_name}_vol_ml", f"{r_name}_voxels"]:
                        if col in row.index:
                            mvols[r_name].append(float(row[col])); break
                    else:
                        mvols[r_name].append(0.0)
    
        print(f"  {mn}: {len(mX)}/{len(keys)} matched to volumes")
        if len(mX) < 10:
            print("    Too few matched — skipping regression tests")
            pids_arr = np.array([k.split("__")[0] for k in keys])
            nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
            _, idx = nbrs.kneighbors(Xs)
            results[mn]["M6_patient_purity_pct"] = float(
                100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                               for i in range(len(keys))]))
            continue
    
        mX = np.stack(mX)
        y_wt = np.array(mvols["wt"])
        y_tc = np.array(mvols["tc"])
        y_et = np.array(mvols["et"])
        ridge = Ridge(alpha=1.0)
        rf    = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    
        # M1: Volume R²
        p_ridge = cross_val_predict(ridge, mX, y_wt, cv=5)
        p_rf    = cross_val_predict(rf,    mX, y_wt, cv=5)
        results[mn]["M1_volume_R2_ridge"]  = float(r2_score(y_wt, p_ridge))
        results[mn]["M1_volume_R2_rf"]     = float(r2_score(y_wt, p_rf))
        rho_ridge = spearmanr(p_ridge, y_wt)[0]
        rho_rf    = spearmanr(p_rf, y_wt)[0]
        # When predictions are near-perfect (R²≈1), Spearman can return NaN due to tied ranks
        # In that case R²≈1 implies perfect rank correlation → use 1.0
        if not np.isnan(rho_ridge):
            rho = rho_ridge
        elif not np.isnan(rho_rf):
            rho = rho_rf
        else:
            r2_best = max(results[mn].get("M1_volume_R2_ridge", 0), results[mn].get("M1_volume_R2_rf", 0))
            rho = 1.0 if r2_best > 0.99 else 0.0
        results[mn]["M1_spearman_rho"]     = float(rho)
    
        # M2: Log-Volume R²
        yl = np.log1p(y_wt)
        results[mn]["M2_logvol_R2_ridge"] = float(r2_score(yl, cross_val_predict(ridge, mX, yl, cv=5)))
        results[mn]["M2_logvol_R2_rf"]    = float(r2_score(yl, cross_val_predict(rf,    mX, yl, cv=5)))
    
        # M3: Enhancement Fraction
        y_ef = y_et / (y_wt + 0.01)
        results[mn]["M3_enhancement_ridge"] = float(r2_score(y_ef, cross_val_predict(ridge, mX, y_ef, cv=5)))
        results[mn]["M3_enhancement_rf"]    = float(r2_score(y_ef, cross_val_predict(rf,    mX, y_ef, cv=5)))
    
        # M4: Necrosis F1
        ncr   = y_tc - y_et
        y_ncr = ((ncr / (y_tc + 1e-6)) > 0.10).astype(int)
        if len(set(y_ncr)) >= 2:
            p_ncr = cross_val_predict(LogisticRegression(max_iter=1000), mX, y_ncr, cv=5)
            results[mn]["M4_necrosis_F1"] = float(f1_score(y_ncr, p_ncr, average="weighted"))
        else:
            results[mn]["M4_necrosis_F1"] = 0.0
    
        # M5: Core Fraction
        y_cf = y_tc / (y_wt + 0.01)
        results[mn]["M5_corefrac_ridge"] = float(r2_score(y_cf, cross_val_predict(ridge, mX, y_cf, cv=5)))
        results[mn]["M5_corefrac_rf"]    = float(r2_score(y_cf, cross_val_predict(rf,    mX, y_cf, cv=5)))
    
        # M6: Patient Identity Purity
        pids_arr = np.array([k.split("__")[0] for k in keys])
        nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
        _, idx = nbrs.kneighbors(Xs)
        results[mn]["M6_patient_purity_pct"] = float(
            100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                           for i in range(len(keys))]))
    
        for k, v in sorted(results[mn].items()):
            if k.startswith("M"):
                print(f"    {mn} {k}: {v:.3f}")

In [ ]:
if not CACHE_LOADED:
    # H1-H5: HETEROGENEITY TESTS
    print("\n" + "=" * 60)
    print("  HETEROGENEITY TESTS H1-H5")
    print("=" * 60)
    
    for mn, embs in models.items():
        keys = list(embs.keys())
        X    = np.stack([embs[k] for k in keys])
        X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        Xs   = StandardScaler().fit_transform(X_l2)
        if mn not in results: results[mn] = {}
    
        # H1 — RankMe (effective dimensionality)
        sub = X_l2[np.random.choice(len(X_l2), min(500, len(X_l2)), replace=False)]
        svs = np.linalg.svd(sub, compute_uv=False)
        p = svs / svs.sum(); p = p[p > 1e-10]
        rankme = float(np.exp(-np.sum(p * np.log(p))))
        results[mn]["H1_rankme"] = rankme
    
        # H2 — Diversity (mean pairwise L2 distance)
        sub500 = X_l2[np.random.choice(len(X_l2), min(500, len(X_l2)), replace=False)]
        dists = [np.linalg.norm(sub500[i] - sub500[j])
                 for i in range(len(sub500)) for j in range(i+1, min(i+20, len(sub500)))]
        results[mn]["H2_diversity"] = float(np.mean(dists))
    
        # H3 — Responder vs Non-responder separability (F1)
        if tumor_df is not None:
            pids_set = set(k.split("__")[0] for k in keys)
            X_resp, y_resp = [], []
            for pid in pids_set:
                pid_keys = sorted([k for k in keys if k.startswith(f"{pid}__")])
                if len(pid_keys) < 2: continue
                stps = [k.split("__")[1] for k in pid_keys]
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[0])]
                vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[-1])]
                if len(v0) > 0 and len(vT) > 0:
                    wt_col = next((c for c in v0.columns if 'wt' in c.lower() and 'vol' in c.lower()), None)
                    if wt_col:
                        ratio = vT.iloc[0][wt_col] / (v0.iloc[0][wt_col] + 1e-6)
                        y_resp.append(1 if ratio < 0.80 else 0)
                        X_resp.append(Xs[keys.index(pid_keys[0])])
            if len(X_resp) >= 10 and len(set(y_resp)) >= 2:
                scores = cross_val_score(LogisticRegression(max_iter=1000),
                                         np.stack(X_resp), y_resp, cv=5, scoring='f1_weighted')
                results[mn]['H3_responder_F1'] = float(scores.mean())
            else:
                results[mn]['H3_responder_F1'] = 0.0
        else:
            results[mn]['H3_responder_F1'] = 0.0
    
        # H4 — Norm CV
        norms = np.linalg.norm(X, axis=1)
        results[mn]['H4_norm_cv'] = float(np.std(norms) / (np.mean(norms) + 1e-8))
    
        # H5 — RankMe standalone
        results[mn]['H5_rankme_standalone'] = rankme
    
        for k, v in sorted(results[mn].items()):
            if k.startswith('H'):
                print(f"    {mn} {k}: {v:.3f}")

In [ ]:
if not CACHE_LOADED:
    # T1-T8: TEMPORAL TESTS
    print('\n' + '='*60)
    print('  TEMPORAL TESTS T1-T8')
    print('='*60)
    from scipy.stats import kendalltau as kt
    
    for mn, embs in models.items():
        keys = list(embs.keys())
        raw  = np.stack([embs[k] for k in keys])
        norms_for_l2 = np.linalg.norm(raw, axis=1, keepdims=True) + 1e-8
        embs_l2 = {k: embs[k] / norms_for_l2[i] for i, k in enumerate(keys)}
        pe   = {}
        for k in keys:
            pid, tp = k.split('__')
            if pid not in pe: pe[pid] = {}
            pe[pid][tp] = embs[k]
        longi = {p: t for p, t in pe.items() if len(t) >= 2}
        print(f'  {mn}: {len(longi)} longitudinal patients')
        if len(longi) < 5:
            for t in ['T1','T2','T3','T4','T5','T6','T7','T8']:
                results[mn][t] = 0
            continue
    
        den, dvol_wt, dvol_et, csim = [], [], [], []
        v_first = []
    
        for pid, tps in longi.items():
            stps = sorted(tps.keys())
            e_first = embs_l2[f'{pid}__{stps[0]}']; e_last = embs_l2[f'{pid}__{stps[-1]}']
            v_first.append(np.linalg.norm(e_last - e_first))
    
            for i in range(len(stps) - 1):
                e0 = embs_l2.get(f'{pid}__{stps[i]}',   tps[stps[i]])
                e1 = embs_l2.get(f'{pid}__{stps[i+1]}', tps[stps[i+1]])
                d = np.linalg.norm(e1 - e0)
                den.append(d)
                n0 = np.linalg.norm(e0) + 1e-8
                n1 = np.linalg.norm(e1) + 1e-8
                csim.append(float((e0/n0) @ (e1/n1)))
                if tumor_df is not None:
                    try:
                        v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[i])]
                        v1 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[i+1])]
                        wt_col = next((c for c in tumor_df.columns if 'wt' in c.lower() and 'vol' in c.lower()), None)
                        et_col = next((c for c in tumor_df.columns if 'et' in c.lower() and 'vol' in c.lower()), None)
                        if len(v0) > 0 and len(v1) > 0 and wt_col:
                            dvol_wt.append(abs(v1.iloc[0][wt_col] - v0.iloc[0][wt_col]))
                            if et_col:
                                et0 = v0.iloc[0][et_col]; et1 = v1.iloc[0][et_col]
                                dvol_et.append(et1 / (et0 + 1e-6) - 1)
                    except Exception:
                        pass
    
        den = np.array(den)
    
        # T1 — Spearman rho
        ml = min(len(den), len(dvol_wt))
        if ml >= 5:
            rho_wt, _ = spearmanr(den[:ml], dvol_wt[:ml])
            results[mn]['T1_spearman_wt'] = abs(float(rho_wt))
        else:
            results[mn]['T1_spearman_wt'] = 0
    
        # T2 — Near-duplicate rate
        near_dup = np.mean(den < 0.001 * den.mean()) if len(den) > 0 else 1.0
        results[mn]['T2_ordering_pass'] = float(near_dup < 0.01)
    
        # T3 — Delta R² + Directional AUC
        ml_et = min(len(den), len(dvol_et))
        if ml_et >= 10:
            yd = np.array(dvol_wt[:ml_et])
            pd3 = cross_val_predict(Ridge(1.0), den[:ml_et].reshape(-1,1), yd, cv=min(5,ml_et//2))
            results[mn]['T3_delta_R2'] = float(r2_score(yd, pd3))
            vol_sign = (np.array(dvol_wt[:ml_et]) > 0).astype(int)
            drift_sign = (den[:ml_et] > np.median(den[:ml_et])).astype(int)
            if len(set(vol_sign)) > 1:
                results[mn]['T3_directional_auc'] = float(roc_auc_score(vol_sign, drift_sign))
            else:
                results[mn]['T3_directional_auc'] = 0.5
        else:
            results[mn]['T3_delta_R2'] = 0
            results[mn]['T3_directional_auc'] = 0.5
    
        # T4 — RANO AUC (ET +40%)
        if ml_et >= 10 and tumor_df is not None:
            progressive = (np.array(dvol_et[:ml_et]) > 0.40).astype(int)
            if len(set(progressive)) > 1:
                results[mn]['T4_rano_auc'] = float(roc_auc_score(progressive, den[:ml_et]))
            else:
                results[mn]['T4_rano_auc'] = 0.5
        else:
            results[mn]['T4_rano_auc'] = 0.5
    
        # T5 — Coherence dual-bound
        coherence = float(np.mean(csim)) if csim else 0
        results[mn]['T5_coherence'] = coherence
        results[mn]['T5_pass_dual'] = float(0.70 < coherence < 0.93)
    
        # T6 — Velocity CV
        results[mn]['T6_velocity_cv'] = float(np.std(den) / (np.mean(den) + 1e-8)) if len(den) else 0
    
        # T7 — Cohen's d (progressors vs stable)
        all_dists = np.array(v_first)
        if tumor_df is not None and len(all_dists) >= 10:
            pids_longi = list(longi.keys())
            prog_mask = []
            for pid in pids_longi:
                stps = sorted(longi[pid].keys())
                et_col = next((c for c in tumor_df.columns if 'et' in c.lower() and 'vol' in c.lower()), None)
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[0])]
                vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[-1])]
                if len(v0) > 0 and len(vT) > 0 and et_col:
                    prog_mask.append((vT.iloc[0][et_col] / (v0.iloc[0][et_col] + 1e-6) - 1) > 0.40)
                else:
                    prog_mask.append(False)
            prog_mask = np.array(prog_mask)
            d_prog = all_dists[prog_mask]; d_stab = all_dists[~prog_mask]
            if len(d_prog) >= 3 and len(d_stab) >= 3:
                pooled = np.sqrt((np.var(d_prog) + np.var(d_stab)) / 2) + 1e-8
                results[mn]['T7_treatment_d'] = float(abs(d_prog.mean() - d_stab.mean()) / pooled)
            else:
                results[mn]['T7_treatment_d'] = 0
        else:
            results[mn]['T7_treatment_d'] = 0
    
        # T8 — Kendall tau trajectory monotonicity
        taus = []
        for pid, tps in longi.items():
            stps = sorted(tps.keys())
            if len(stps) < 2: continue
            dists = [np.linalg.norm(tps[v] - tps[stps[0]]) for v in stps[1:]]
            tau, _ = kt(dists, range(len(dists)))
            taus.append(tau)
        results[mn]['T8_kendall_tau'] = float(np.nanmean(taus)) if taus else 0
    
        for k, v in sorted(results[mn].items()):
            if k.startswith('T'):
                flag = ' <- WEAK' if v < 0.30 and k in [
                    'T1_spearman_wt','T3_delta_R2','T3_directional_auc','T4_rano_auc','T8_kendall_tau'] else ''
                print(f'    {mn} {k}: {v:.3f}{flag}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ABLATION: Full 18-Test Battery WITHOUT 9D Volumetric Features
# ═══════════════════════════════════════════════════════════
# Tests whether neural features alone (octant 2048D + region 768D = 2816D)
# carry morphological & temporal signal, or if it's all in the 9D volume tail.

from scipy.stats import kendalltau as kt

VOL_DIM = 9  # last 9 dims = volumetric features

# Check if ablation was already cached
try:
    _skip_abl = CACHE_LOADED and len(results_ablation) > 0
except NameError:
    results_ablation = {}
    _skip_abl = False

if _skip_abl:
    print("=" * 60)
    print("  ABLATION (No-Vol): LOADED FROM CACHE")
    print("=" * 60)
    for mn in results_ablation:
        print(f"  {mn}: {len(results_ablation[mn])} metrics")

if not _skip_abl:
    print("=" * 60)
    print("  ABLATION: No-Volumetric (2816D neural features only)")
    print("=" * 60)

    results_ablation = {}

    for mn, embs in models.items():
        keys = list(embs.keys())
        X_full = np.stack([embs[k] for k in keys])
    
        # ── STRIP the last 9D (volumetric) ──
        X_no_vol = X_full[:, :-VOL_DIM]
        D_full = X_full.shape[1]
        D_nv   = X_no_vol.shape[1]
        print(f"\n  {mn}: {D_full}D → {D_nv}D (removed {VOL_DIM}D volumetric)")
    
        X_l2 = X_no_vol / (np.linalg.norm(X_no_vol, axis=1, keepdims=True) + 1e-8)
        Xs   = StandardScaler().fit_transform(X_l2)
        results_ablation[mn] = {}

        # ── M1-M6: MORPHOLOGY ──
        mX, mvols = [], {"wt": [], "tc": [], "et": []}
        for k in keys:
            pid, tp = k.split("__")
            row = match_row(tumor_df, pid, tp)
            if row is not None:
                mX.append(Xs[keys.index(k)])
                for r_name in ["wt", "tc", "et"]:
                    for col in [f"{r_name}_vol", f"{r_name.upper()}_vol",
                                 f"{r_name}_volume", f"vol_{r_name}",
                                 f"{r_name}_vol_ml", f"{r_name}_voxels"]:
                        if col in row.index:
                            mvols[r_name].append(float(row[col])); break
                    else:
                        mvols[r_name].append(0.0)

        if len(mX) >= 10:
            mX = np.stack(mX)
            y_wt = np.array(mvols["wt"])
            y_tc = np.array(mvols["tc"])
            y_et = np.array(mvols["et"])
            ridge = Ridge(alpha=1.0)
            rf    = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

            p_ridge = cross_val_predict(ridge, mX, y_wt, cv=5)
            p_rf    = cross_val_predict(rf,    mX, y_wt, cv=5)
            results_ablation[mn]["M1_volume_R2_ridge"]  = float(r2_score(y_wt, p_ridge))
            results_ablation[mn]["M1_volume_R2_rf"]     = float(r2_score(y_wt, p_rf))
            rho_ridge = spearmanr(p_ridge, y_wt)[0]
            rho_rf    = spearmanr(p_rf, y_wt)[0]
            if not np.isnan(rho_ridge):
                rho = rho_ridge
            elif not np.isnan(rho_rf):
                rho = rho_rf
            else:
                r2_best = max(results_ablation[mn].get("M1_volume_R2_ridge", 0), results_ablation[mn].get("M1_volume_R2_rf", 0))
                rho = 1.0 if r2_best > 0.99 else 0.0
            results_ablation[mn]["M1_spearman_rho"] = float(rho)

            yl = np.log1p(y_wt)
            results_ablation[mn]["M2_logvol_R2_ridge"] = float(r2_score(yl, cross_val_predict(ridge, mX, yl, cv=5)))
            results_ablation[mn]["M2_logvol_R2_rf"]    = float(r2_score(yl, cross_val_predict(rf,    mX, yl, cv=5)))

            y_ef = y_et / (y_wt + 0.01)
            results_ablation[mn]["M3_enhancement_ridge"] = float(r2_score(y_ef, cross_val_predict(ridge, mX, y_ef, cv=5)))
            results_ablation[mn]["M3_enhancement_rf"]    = float(r2_score(y_ef, cross_val_predict(rf,    mX, y_ef, cv=5)))

            ncr   = y_tc - y_et
            y_ncr = ((ncr / (y_tc + 1e-6)) > 0.10).astype(int)
            if len(set(y_ncr)) >= 2:
                p_ncr = cross_val_predict(LogisticRegression(max_iter=1000), mX, y_ncr, cv=5)
                results_ablation[mn]["M4_necrosis_F1"] = float(f1_score(y_ncr, p_ncr, average="weighted"))
            else:
                results_ablation[mn]["M4_necrosis_F1"] = 0.0

            y_cf = y_tc / (y_wt + 0.01)
            results_ablation[mn]["M5_corefrac_ridge"] = float(r2_score(y_cf, cross_val_predict(ridge, mX, y_cf, cv=5)))
            results_ablation[mn]["M5_corefrac_rf"]    = float(r2_score(y_cf, cross_val_predict(rf,    mX, y_cf, cv=5)))

        pids_arr = np.array([k.split("__")[0] for k in keys])
        nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
        _, idx = nbrs.kneighbors(Xs)
        results_ablation[mn]["M6_patient_purity_pct"] = float(
            100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                           for i in range(len(keys))]))

        # ── H1-H5: HETEROGENEITY ──
        sub = X_l2[np.random.choice(len(X_l2), min(500, len(X_l2)), replace=False)]
        svs = np.linalg.svd(sub, compute_uv=False)
        p = svs / svs.sum(); p = p[p > 1e-10]
        rankme = float(np.exp(-np.sum(p * np.log(p))))
        results_ablation[mn]["H1_rankme"] = rankme

        sub500 = X_l2[np.random.choice(len(X_l2), min(500, len(X_l2)), replace=False)]
        dists_h = [np.linalg.norm(sub500[i] - sub500[j])
                   for i in range(len(sub500)) for j in range(i+1, min(i+20, len(sub500)))]
        results_ablation[mn]["H2_diversity"] = float(np.mean(dists_h))

        if tumor_df is not None:
            pids_set = set(k.split("__")[0] for k in keys)
            X_resp, y_resp = [], []
            for pid in pids_set:
                pid_keys = sorted([k for k in keys if k.startswith(f"{pid}__")])
                if len(pid_keys) < 2: continue
                stps = [k.split("__")[1] for k in pid_keys]
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[0])]
                vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[-1])]
                if len(v0) > 0 and len(vT) > 0:
                    wt_col = next((c for c in v0.columns if 'wt' in c.lower() and 'vol' in c.lower()), None)
                    if wt_col:
                        ratio = vT.iloc[0][wt_col] / (v0.iloc[0][wt_col] + 1e-6)
                        y_resp.append(1 if ratio < 0.80 else 0)
                        X_resp.append(Xs[keys.index(pid_keys[0])])
            if len(X_resp) >= 10 and len(set(y_resp)) >= 2:
                scores = cross_val_score(LogisticRegression(max_iter=1000),
                                         np.stack(X_resp), y_resp, cv=5, scoring='f1_weighted')
                results_ablation[mn]['H3_responder_F1'] = float(scores.mean())
            else:
                results_ablation[mn]['H3_responder_F1'] = 0.0

        norms = np.linalg.norm(X_no_vol, axis=1)
        results_ablation[mn]['H4_norm_cv'] = float(np.std(norms) / (np.mean(norms) + 1e-8))
        results_ablation[mn]['H5_rankme_standalone'] = rankme

        # ── T1-T8: TEMPORAL ──
        raw = X_no_vol
        norms_t = np.linalg.norm(raw, axis=1, keepdims=True) + 1e-8
        embs_l2_nv = {k: raw[i] / norms_t[i] for i, k in enumerate(keys)}
        pe = {}
        for k in keys:
            pid, tp = k.split('__')
            if pid not in pe: pe[pid] = {}
            pe[pid][tp] = raw[i]
        # Rebuild pe properly
        pe = {}
        for i, k in enumerate(keys):
            pid, tp = k.split('__')
            if pid not in pe: pe[pid] = {}
            pe[pid][tp] = raw[i]
        longi = {p: t for p, t in pe.items() if len(t) >= 2}

        den, dvol_wt, dvol_et, csim = [], [], [], []
        v_first = []

        for pid, tps in longi.items():
            stps = sorted(tps.keys())
            e_first = embs_l2_nv[f'{pid}__{stps[0]}']; e_last = embs_l2_nv[f'{pid}__{stps[-1]}']
            v_first.append(np.linalg.norm(e_last - e_first))
            for i in range(len(stps) - 1):
                e0 = embs_l2_nv.get(f'{pid}__{stps[i]}',   tps[stps[i]])
                e1 = embs_l2_nv.get(f'{pid}__{stps[i+1]}', tps[stps[i+1]])
                d = np.linalg.norm(e1 - e0)
                den.append(d)
                n0 = np.linalg.norm(e0) + 1e-8; n1 = np.linalg.norm(e1) + 1e-8
                csim.append(float((e0/n0) @ (e1/n1)))
                if tumor_df is not None:
                    try:
                        v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[i])]
                        v1 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[i+1])]
                        wt_col = next((c for c in tumor_df.columns if 'wt' in c.lower() and 'vol' in c.lower()), None)
                        et_col = next((c for c in tumor_df.columns if 'et' in c.lower() and 'vol' in c.lower()), None)
                        if len(v0) > 0 and len(v1) > 0 and wt_col:
                            dvol_wt.append(abs(v1.iloc[0][wt_col] - v0.iloc[0][wt_col]))
                            if et_col:
                                et0 = v0.iloc[0][et_col]; et1 = v1.iloc[0][et_col]
                                dvol_et.append(et1 / (et0 + 1e-6) - 1)
                    except Exception:
                        pass

        den = np.array(den)
        ml = min(len(den), len(dvol_wt))
        results_ablation[mn]['T1_spearman_wt'] = abs(float(spearmanr(den[:ml], dvol_wt[:ml])[0])) if ml >= 5 else 0
        near_dup = np.mean(den < 0.001 * den.mean()) if len(den) > 0 else 1.0
        results_ablation[mn]['T2_ordering_pass'] = float(near_dup < 0.01)

        ml_et = min(len(den), len(dvol_et))
        if ml_et >= 10:
            yd = np.array(dvol_wt[:ml_et])
            pd3 = cross_val_predict(Ridge(1.0), den[:ml_et].reshape(-1,1), yd, cv=min(5,ml_et//2))
            results_ablation[mn]['T3_delta_R2'] = float(r2_score(yd, pd3))
            vol_sign = (np.array(dvol_wt[:ml_et]) > 0).astype(int)
            drift_sign = (den[:ml_et] > np.median(den[:ml_et])).astype(int)
            if len(set(vol_sign)) > 1:
                results_ablation[mn]['T3_directional_auc'] = float(roc_auc_score(vol_sign, drift_sign))
            else:
                results_ablation[mn]['T3_directional_auc'] = 0.5
        else:
            results_ablation[mn]['T3_delta_R2'] = 0; results_ablation[mn]['T3_directional_auc'] = 0.5

        if ml_et >= 10 and tumor_df is not None:
            progressive = (np.array(dvol_et[:ml_et]) > 0.40).astype(int)
            if len(set(progressive)) > 1:
                results_ablation[mn]['T4_rano_auc'] = float(roc_auc_score(progressive, den[:ml_et]))
            else:
                results_ablation[mn]['T4_rano_auc'] = 0.5
        else:
            results_ablation[mn]['T4_rano_auc'] = 0.5

        coherence = float(np.mean(csim)) if csim else 0
        results_ablation[mn]['T5_coherence'] = coherence
        results_ablation[mn]['T5_pass_dual'] = float(0.70 < coherence < 0.93)
        results_ablation[mn]['T6_velocity_cv'] = float(np.std(den) / (np.mean(den) + 1e-8)) if len(den) else 0

        all_dists = np.array(v_first)
        if tumor_df is not None and len(all_dists) >= 10:
            pids_longi = list(longi.keys())
            prog_mask = []
            for pid in pids_longi:
                stps = sorted(longi[pid].keys())
                et_col = next((c for c in tumor_df.columns if 'et' in c.lower() and 'vol' in c.lower()), None)
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[0])]
                vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[-1])]
                if len(v0) > 0 and len(vT) > 0 and et_col:
                    prog_mask.append((vT.iloc[0][et_col] / (v0.iloc[0][et_col] + 1e-6) - 1) > 0.40)
                else:
                    prog_mask.append(False)
            prog_mask = np.array(prog_mask)
            d_prog = all_dists[prog_mask]; d_stab = all_dists[~prog_mask]
            if len(d_prog) >= 3 and len(d_stab) >= 3:
                pooled = np.sqrt((np.var(d_prog) + np.var(d_stab)) / 2) + 1e-8
                results_ablation[mn]['T7_treatment_d'] = float(abs(d_prog.mean() - d_stab.mean()) / pooled)
            else:
                results_ablation[mn]['T7_treatment_d'] = 0
        else:
            results_ablation[mn]['T7_treatment_d'] = 0

        taus = []
        for pid, tps in longi.items():
            stps = sorted(tps.keys())
            if len(stps) < 2: continue
            dists_t = [np.linalg.norm(tps[v] - tps[stps[0]]) for v in stps[1:]]
            tau, _ = kt(dists_t, range(len(dists_t)))
            taus.append(tau)
        results_ablation[mn]['T8_kendall_tau'] = float(np.nanmean(taus)) if taus else 0

    # ── COMPARISON TABLE ──
    print("\n" + "=" * 80)
    print("  ABLATION COMPARISON: Full (2825D) vs No-Vol (2816D)")
    print("=" * 80)

    KEY_TESTS = [
        "M1_volume_R2_rf", "M1_spearman_rho",
        "M2_logvol_R2_rf", "M3_enhancement_rf",
        "M4_necrosis_F1", "M5_corefrac_rf",
        "M6_patient_purity_pct",
        "H1_rankme", "H2_diversity", "H3_responder_F1", "H4_norm_cv",
        "T1_spearman_wt", "T2_ordering_pass", "T3_directional_auc",
        "T4_rano_auc", "T5_coherence", "T7_treatment_d", "T8_kendall_tau",
    ]

    for mn in models:
        r_full = results.get(mn, {})
        r_nv   = results_ablation.get(mn, {})
        print(f"\n  Model: {mn}")
        print(f"  {'Test':<30} {'Full(2825D)':>12} {'NoVol(2816D)':>14} {'Δ':>8} {'Signal?':>10}")
        print("  " + "-" * 76)
        for t in KEY_TESTS:
            vf = r_full.get(t, 0)
            vn = r_nv.get(t, 0)
            delta = vn - vf
            # Determine if vol contributed positively or neural features carry signal
            if abs(delta) < 0.01:
                note = "≈ same"
            elif delta > 0:
                note = "↑ neural"
            else:
                note = "↓ vol helps"
            print(f"  {t:<30} {vf:>12.3f} {vn:>14.3f} {delta:>+8.3f} {note:>10}")

    print("\n  Legend: ↑ neural = removing vol IMPROVED score (neural features alone are better)")
    print("         ↓ vol helps = removing vol HURT score (vol features contributed)")
    print("         ≈ same = negligible difference (<0.01)")

    # DIAGNOSTIC: Check if R2=1.000 is genuine or a bug
    print("\n" + "=" * 80)
    print("  DIAGNOSTIC: R2 Precision Check + Permutation Baseline")
    print("=" * 80)
    for mn in models:
        r_nv = results_ablation.get(mn, {})
        keys_m = list(models[mn].keys())
        X_full_d = np.stack([models[mn][k] for k in keys_m])
        X_nv_d = X_full_d[:, :-VOL_DIM]
        X_l2_d = X_nv_d / (np.linalg.norm(X_nv_d, axis=1, keepdims=True) + 1e-8)
        Xs_d = StandardScaler().fit_transform(X_l2_d)
        mX_d, y_wt_d = [], []
        for k in keys_m:
            pid, tp = k.split("__")
            row = match_row(tumor_df, pid, tp)
            if row is not None:
                mX_d.append(Xs_d[keys_m.index(k)])
                for col in ["wt_vol", "WT_vol", "wt_volume", "vol_wt", "wt_vol_ml", "wt_voxels"]:
                    if col in row.index:
                        y_wt_d.append(float(row[col])); break
                else:
                    y_wt_d.append(0.0)
        mX_d = np.stack(mX_d)
        y_wt_d = np.array(y_wt_d)
        rf_d = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        p_d = cross_val_predict(rf_d, mX_d, y_wt_d, cv=5)
        r2_actual = r2_score(y_wt_d, p_d)
        print(f"  {mn} No-Vol R2 (6 dec): {r2_actual:.6f}")
        np.random.seed(42)
        y_shuffled = y_wt_d.copy()
        np.random.shuffle(y_shuffled)
        p_shuf = cross_val_predict(rf_d, mX_d, y_shuffled, cv=5)
        r2_shuf = r2_score(y_shuffled, p_shuf)
        print(f"  {mn} Shuffled R2 (sanity): {r2_shuf:.6f}")
        if r2_shuf < 0.1:
            print(f"    -> CV works. Features genuinely encode volume info.")
        else:
            print(f"    -> WARNING: Shuffled R2 too high, possible CV leak!")
        rf_small = RandomForestRegressor(n_estimators=20, max_depth=5, random_state=42, n_jobs=-1)
        p_small = cross_val_predict(rf_small, mX_d, y_wt_d, cv=5)
        r2_small = r2_score(y_wt_d, p_small)
        print(f"  {mn} Reduced RF (20t, d=5): {r2_small:.6f}")
        ridge_d = Ridge(alpha=1.0)
        p_ridge_d = cross_val_predict(ridge_d, mX_d, y_wt_d, cv=5)
        r2_ridge = r2_score(y_wt_d, p_ridge_d)
        print(f"  {mn} Ridge (linear) R2: {r2_ridge:.6f}")
        print(f"  Volume stats: min={y_wt_d.min():.0f} max={y_wt_d.max():.0f} mean={y_wt_d.mean():.0f}")
        print(f"  Embedding shape: {mX_d.shape}")

    # Save ablation results
    abl_out = OUTPUT_ROOT / 'm2_ablation_results.json'
    import json as _json
    with open(abl_out, 'w') as f:
        _json.dump(results_ablation, f, indent=2)
    print(f"\n  Saved: {abl_out}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# ABLATION: Per-Segment Evaluation
# ═══════════════════════════════════════════════════════════
# Embedding = octant(2048D) + region(768D) + vol(9D)
# nnUNet: 2048 + 768 + 9 = 2825D (C=256)
# SwinUNETR: 3072 + 1152 + 9 = 4233D (C=384)
# C = (D - 9) // 11 adapts automatically.

from scipy.stats import kendalltau as kt

# Check if segment ablation was already cached
try:
    _skip_seg = CACHE_LOADED and len(results_seg) > 0
except NameError:
    results_seg = {}
    _skip_seg = False

if _skip_seg:
    print("=" * 80)
    print("  PER-SEGMENT ABLATION: LOADED FROM CACHE")
    print("=" * 80)
    for mn in results_seg:
        print(f"  {mn}: {len(results_seg[mn])} segments cached")

if not _skip_seg:
    print("=" * 80)
    print("  PER-SEGMENT ABLATION")
    print("=" * 80)

    results_seg = {}

    for mn, embs in models.items():
        keys = list(embs.keys())
        X_full = np.stack([embs[k] for k in keys])
        D = X_full.shape[1]
        C = (D - 9) // 11
        oct_d, reg_d, vol_d = 8 * C, 3 * C, 9

        # Define segments
        segments = {
            "octant_only":   X_full[:, :oct_d],
            "region_only":   X_full[:, oct_d:oct_d+reg_d],
            "vol_only":      X_full[:, oct_d+reg_d:],
            "oct+reg":       X_full[:, :oct_d+reg_d],
            "oct+vol":       np.concatenate([X_full[:, :oct_d], X_full[:, oct_d+reg_d:]], axis=1),
            "reg+vol":       X_full[:, oct_d:],
            "full":          X_full,
        }

        seg_dims = {k: v.shape[1] for k, v in segments.items()}
        print(f"\n  {mn} | C={C} | Segments: " + ", ".join(f"{k}={v}D" for k, v in seg_dims.items()))

        results_seg[mn] = {}

        for seg_name, X_seg in segments.items():
            X_l2 = X_seg / (np.linalg.norm(X_seg, axis=1, keepdims=True) + 1e-8)
            Xs = StandardScaler().fit_transform(X_l2)

            # ── MORPHOLOGY: M1 Volume R² + M3 Enhancement + M5 Core Fraction ──
            mX, mvols = [], {"wt": [], "tc": [], "et": []}
            mk = []
            for k in keys:
                pid, tp = k.split("__")
                row = match_row(tumor_df, pid, tp)
                if row is not None:
                    mX.append(Xs[keys.index(k)])
                    mk.append(k)
                    for r_name in ["wt", "tc", "et"]:
                        for col in [f"{r_name}_vol", f"{r_name.upper()}_vol",
                                     f"{r_name}_volume", f"vol_{r_name}",
                                     f"{r_name}_vol_ml", f"{r_name}_voxels"]:
                            if col in row.index:
                                mvols[r_name].append(float(row[col])); break
                        else:
                            mvols[r_name].append(0.0)

            sr = {}  # segment results

            if len(mX) >= 10:
                mX = np.stack(mX)
                y_wt = np.array(mvols["wt"])
                y_tc = np.array(mvols["tc"])
                y_et = np.array(mvols["et"])

                ridge = Ridge(alpha=1.0)
                rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

                # M1: Volume R²
                p_rf = cross_val_predict(rf, mX, y_wt, cv=5)
                p_ridge_v = cross_val_predict(ridge, mX, y_wt, cv=5)
                sr["M1_vol_R2_rf"] = float(r2_score(y_wt, p_rf))
                sr["M1_vol_R2_ridge"] = float(r2_score(y_wt, p_ridge_v))
                rho_val = spearmanr(p_rf, y_wt)[0]
                sr["M1_spearman"] = float(rho_val) if not np.isnan(rho_val) else 1.0

                # M3: Enhancement fraction
                y_ef = y_et / (y_wt + 0.01)
                p_ef = cross_val_predict(rf, mX, y_ef, cv=5)
                sr["M3_enhance_R2"] = float(r2_score(y_ef, p_ef))

                # M5: Core fraction
                y_cf = y_tc / (y_wt + 0.01)
                p_cf = cross_val_predict(rf, mX, y_cf, cv=5)
                sr["M5_core_R2"] = float(r2_score(y_cf, p_cf))
            else:
                sr["M1_vol_R2_rf"] = 0
                sr["M1_vol_R2_ridge"] = 0
                sr["M1_spearman"] = 0
                sr["M3_enhance_R2"] = 0
                sr["M5_core_R2"] = 0

            # ── HETEROGENEITY: H1 RankMe + H3 Responder F1 ──
            sub = X_l2[np.random.choice(len(X_l2), min(500, len(X_l2)), replace=False)]
            svs = np.linalg.svd(sub, compute_uv=False)
            p_sv = svs / svs.sum(); p_sv = p_sv[p_sv > 1e-10]
            sr["H1_rankme"] = float(np.exp(-np.sum(p_sv * np.log(p_sv))))

            if tumor_df is not None:
                pids_set = set(k.split("__")[0] for k in keys)
                X_resp, y_resp = [], []
                for pid in pids_set:
                    pid_keys = sorted([k for k in keys if k.startswith(f"{pid}__")])
                    if len(pid_keys) < 2: continue
                    stps = [k.split("__")[1] for k in pid_keys]
                    v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[0])]
                    vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[-1])]
                    if len(v0) > 0 and len(vT) > 0:
                        wt_col = next((c for c in v0.columns if 'wt' in c.lower() and 'vol' in c.lower()), None)
                        if wt_col:
                            ratio = vT.iloc[0][wt_col] / (v0.iloc[0][wt_col] + 1e-6)
                            y_resp.append(1 if ratio < 0.80 else 0)
                            X_resp.append(Xs[keys.index(pid_keys[0])])
                if len(X_resp) >= 10 and len(set(y_resp)) >= 2:
                    scores = cross_val_score(LogisticRegression(max_iter=1000),
                                             np.stack(X_resp), y_resp, cv=5, scoring='f1_weighted')
                    sr["H3_responder"] = float(scores.mean())
                else:
                    sr["H3_responder"] = 0.0
            else:
                sr["H3_responder"] = 0.0

            # M6: Patient purity
            pids_arr = np.array([k.split("__")[0] for k in keys])
            nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
            _, idx = nbrs.kneighbors(Xs)
            sr["M6_purity"] = float(
                100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                               for i in range(len(keys))]))

            # ── TEMPORAL: T1 Spearman(drift,dvol) + T5 Coherence + T8 Kendall ──
            norms_t = np.linalg.norm(X_seg, axis=1, keepdims=True) + 1e-8
            embs_l2_s = {k: X_seg[i] / norms_t[i] for i, k in enumerate(keys)}
            pe = {}
            for i_k, k in enumerate(keys):
                pid, tp = k.split('__')
                if pid not in pe: pe[pid] = {}
                pe[pid][tp] = X_seg[i_k]
            longi = {p: t for p, t in pe.items() if len(t) >= 2}

            den, dvol_wt, csim = [], [], []
            for pid, tps in longi.items():
                stps = sorted(tps.keys())
                for i_t in range(len(stps) - 1):
                    e0 = embs_l2_s.get(f'{pid}__{stps[i_t]}', tps[stps[i_t]])
                    e1 = embs_l2_s.get(f'{pid}__{stps[i_t+1]}', tps[stps[i_t+1]])
                    d = np.linalg.norm(e1 - e0)
                    den.append(d)
                    n0 = np.linalg.norm(e0) + 1e-8; n1 = np.linalg.norm(e1) + 1e-8
                    csim.append(float((e0/n0) @ (e1/n1)))
                    if tumor_df is not None:
                        try:
                            v0r = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[i_t])]
                            v1r = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint'].astype(str)==stps[i_t+1])]
                            wt_col = next((c for c in tumor_df.columns if 'wt' in c.lower() and 'vol' in c.lower()), None)
                            if len(v0r) > 0 and len(v1r) > 0 and wt_col:
                                dvol_wt.append(abs(v1r.iloc[0][wt_col] - v0r.iloc[0][wt_col]))
                        except Exception:
                            pass

            den = np.array(den) if den else np.array([0])
            ml = min(len(den), len(dvol_wt))
            sr["T1_spearman"] = abs(float(spearmanr(den[:ml], dvol_wt[:ml])[0])) if ml >= 5 else 0
            sr["T5_coherence"] = float(np.mean(csim)) if csim else 0

            taus = []
            for pid, tps in longi.items():
                stps = sorted(tps.keys())
                if len(stps) < 2: continue
                dists_t = [np.linalg.norm(tps[v] - tps[stps[0]]) for v in stps[1:]]
                tau, _ = kt(dists_t, range(len(dists_t)))
                taus.append(tau)
            sr["T8_kendall"] = float(np.nanmean(taus)) if taus else 0

            results_seg[mn][seg_name] = sr

        # ── PRINT COMPARISON TABLE ──
        seg_names = list(segments.keys())
        key_metrics = ["M1_vol_R2_rf", "M1_vol_R2_ridge", "M1_spearman",
                       "M3_enhance_R2", "M5_core_R2", "M6_purity",
                       "H1_rankme", "H3_responder",
                       "T1_spearman", "T5_coherence", "T8_kendall"]

        print(f"\n  {'Metric':<22}", end="")
        for sn in seg_names:
            d = seg_dims[sn]
            print(f" {sn+'('+str(d)+')':>16}", end="")
        print()
        print("  " + "-" * (22 + 17 * len(seg_names)))

        for metric in key_metrics:
            print(f"  {metric:<22}", end="")
            vals = []
            for sn in seg_names:
                v = results_seg[mn][sn].get(metric, 0)
                vals.append(v)
                print(f" {v:>16.3f}", end="")
            # Mark best
            best_idx = np.argmax(vals)
            print(f"  <- {seg_names[best_idx]}")

        # ── INTERPRETATION ──
        full_r2 = results_seg[mn]["full"]["M1_vol_R2_rf"]
        oct_r2 = results_seg[mn]["octant_only"]["M1_vol_R2_rf"]
        reg_r2 = results_seg[mn]["region_only"]["M1_vol_R2_rf"]
        vol_r2 = results_seg[mn]["vol_only"]["M1_vol_R2_rf"]
        no_vol_r2 = results_seg[mn]["oct+reg"]["M1_vol_R2_rf"]

        print(f"\n  INTERPRETATION:")
        print(f"    vol_only R2={vol_r2:.3f}: 9D volume features {'DO' if vol_r2 > 0.5 else 'do NOT'} encode volume (expected: high)")
        print(f"    octant R2={oct_r2:.3f}: spatial features {'DO' if oct_r2 > 0.5 else 'do NOT'} encode volume")
        print(f"    region R2={reg_r2:.3f}: mask-weighted features {'DO' if reg_r2 > 0.5 else 'do NOT'} encode volume")
        print(f"    oct+reg R2={no_vol_r2:.3f}: neural features {'DO' if no_vol_r2 > 0.5 else 'do NOT'} encode volume")

        if vol_r2 > 0.9 and no_vol_r2 < 0.5:
            print(f"    -> Volume signal comes ONLY from the 9D explicit features (good design)")
        elif vol_r2 > 0.9 and no_vol_r2 > 0.5:
            print(f"    -> Volume is encoded in BOTH neural + explicit features (redundant)")
        elif vol_r2 < 0.5:
            print(f"    -> WARNING: vol_only doesn't predict volume — check extraction!")

        # Temporal comparison
        oct_t8 = results_seg[mn]["octant_only"]["T8_kendall"]
        reg_t8 = results_seg[mn]["region_only"]["T8_kendall"]
        vol_t8 = results_seg[mn]["vol_only"]["T8_kendall"]
        print(f"\n    Temporal monotonicity (T8 Kendall tau):")
        print(f"      octant={oct_t8:.3f}  region={reg_t8:.3f}  vol={vol_t8:.3f}")
        if oct_t8 > reg_t8 and oct_t8 > vol_t8:
            print(f"      -> Octant features carry best temporal signal")
        elif reg_t8 > oct_t8 and reg_t8 > vol_t8:
            print(f"      -> Region features carry best temporal signal")
        else:
            print(f"      -> Volume features carry best temporal signal")

    # Save segment results
    seg_out = OUTPUT_ROOT / 'm2_segment_ablation.json'
    import json as _json
    with open(seg_out, 'w') as f:
        _json.dump(results_seg, f, indent=2)
    print(f"\n  Saved: {seg_out}")


In [ ]:
# FULL 18-TEST DASHBOARD + SAVE RESULTS
print("\n" + "="*60)
print("  FULL 18-TEST DASHBOARD — MU-Glioma-Post")
print("="*60)

LOWER_BETTER = {"T2_ordering_pass": False, "H4_norm_cv": True}

THRESHOLDS = {
    "M1_volume_R2_ridge":    None,
    "M1_volume_R2_rf":       None,
    "M1_spearman_rho":       (0.55, "HIGH-PRI"),
    "M2_logvol_R2_ridge":    None,
    "M2_logvol_R2_rf":       None,
    "M3_enhancement_ridge":  None,
    "M3_enhancement_rf":     None,
    "M4_necrosis_F1":        (0.60, "MEDIUM"),
    "M5_corefrac_ridge":     None,
    "M5_corefrac_rf":        None,
    "M6_patient_purity_pct": (60.0, "HIGH-PRI"),
    "H1_rankme":             (30.0, "MEDIUM"),
    "H2_diversity":          (0.25, "MEDIUM"),
    "H3_responder_F1":       (0.55, "HIGH-PRI"),
    "H4_norm_cv":            (0.50, "LOW"),
    "H5_rankme_standalone":  None,
    "T1_spearman_wt":        (0.25, "MEDIUM"),
    "T2_ordering_pass":      (1.0,  "HIGH-PRI"),
    "T3_delta_R2":           None,
    "T3_directional_auc":    (0.55, "MEDIUM"),
    "T4_rano_auc":           (0.55, "MEDIUM"),
    "T5_coherence":          None,
    "T5_pass_dual":          (1.0,  "MEDIUM"),
    "T6_velocity_cv":        None,
    "T7_treatment_d":        (0.30, "MEDIUM"),
    "T8_kendall_tau":        (0.20, "MEDIUM"),
}

for mn in models:
    r = results.get(mn, {})
    passed = total = 0
    high_pri_fails = []
    for k, tup in THRESHOLDS.items():
        if tup is None or k not in r: continue
        thresh, pri = tup
        v = r[k]
        total += 1
        is_nan = isinstance(v, float) and (v != v)
        ok = False if is_nan else ((v <= thresh) if k in LOWER_BETTER else (v >= thresh))
        if ok: passed += 1
        if pri == "HIGH-PRI" and not ok:
            high_pri_fails.append((k, v, thresh, is_nan))
    print(f"\n{mn}  [{passed}/{total} pass]")
    for k, tup in THRESHOLDS.items():
        if tup is None:
            v = r.get(k)
            if v is not None: print(f"  {k:<35s} {v:>8.3f}  (descriptor)")
            continue
        if k not in r: continue
        thresh, pri = tup
        v = r[k]
        is_nan = isinstance(v, float) and (v != v)
        if is_nan:
            print(f"  {k:<35s} {'nan':>8s}  thresh={str(thresh):<6s}  FAIL  [{pri}]")
            continue
        ok = (v <= thresh) if k in LOWER_BETTER else (v >= thresh)
        flag = "PASS" if ok else "FAIL"
        print(f"  {k:<35s} {v:>8.3f}  thresh={str(thresh):<6s}  {flag}  [{pri}]")
    if high_pri_fails:
        print("\n  HIGH-PRIORITY FAILS:")
        for k, v, t, is_nan in high_pri_fails:
            if is_nan:
                print(f"    {k} = NaN")
            else:
                print(f"    {k} = {v:.3f} -> needs > {t:.2f}")

# ── Save JSON ──
json_path = OUTPUT_ROOT / "m2_eval_results.json"
serializable = {}
for mn, r in results.items():
    serializable[mn] = {k: (None if (isinstance(v, float) and v!=v) else
                             float(v) if isinstance(v, (np.floating, float)) else v)
                        for k, v in r.items()}
with open(json_path, "w") as f:
    _json.dump(serializable, f, indent=2)
print(f"\nSaved: {json_path}")

# ── Save ablation results ──
try:
    if results_ablation:
        abl_path = OUTPUT_ROOT / "m2_ablation_results.json"
        ser_abl = {}
        for mn, r in results_ablation.items():
            ser_abl[mn] = {k: (None if (isinstance(v, float) and v!=v) else
                             float(v) if isinstance(v, (np.floating, float)) else v)
                          for k, v in r.items()}
        with open(abl_path, "w") as f:
            _json.dump(ser_abl, f, indent=2)
        print(f"Saved: {abl_path}")
except NameError:
    pass

try:
    if results_seg:
        seg_path = OUTPUT_ROOT / "m2_segment_ablation.json"
        ser_seg = {}
        for mn, segs in results_seg.items():
            ser_seg[mn] = {}
            for seg_name, metrics in segs.items():
                ser_seg[mn][seg_name] = {k: (None if (isinstance(v, float) and v!=v) else
                                             float(v) if isinstance(v, (np.floating, float)) else v)
                                        for k, v in metrics.items()}
        with open(seg_path, "w") as f:
            _json.dump(ser_seg, f, indent=2)
        print(f"Saved: {seg_path}")
except NameError:
    pass

# ── t-SNE + Plots ──
if CACHE_LOADED:
    print("\n⚠ t-SNE skipped in cache mode")
else:
    print("\nGenerating t-SNE visualizations...")
    for mn, emb_dict in models.items():
        if not emb_dict: continue
        keys = sorted(emb_dict.keys())
        X = np.stack([emb_dict[k] for k in keys])
        X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        pids = [k.split("__")[0] for k in keys]
        tps  = [int(k.split("__")[1]) for k in keys]

        X_pca50 = PCA(n_components=min(50, X_norm.shape[1])).fit_transform(X_norm)
        X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pca50)

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        sc = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=tps, cmap="viridis", s=8, alpha=0.6)
        plt.colorbar(sc, ax=axes[0], label="Timepoint")
        axes[0].set_title(f"{mn} — t-SNE by timepoint")

        if tumor_df is not None:
            wt_col = next((c for c in tumor_df.columns if "wt" in c.lower() and "vol" in c.lower()), None)
            vols = []
            for k in keys:
                pid, tp = k.split("__")
                row = match_row(tumor_df, pid, tp)
                vols.append(float(row[wt_col]) if (row is not None and wt_col) else 0)
            sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                                  c=np.log1p(vols), cmap="hot", s=8, alpha=0.6)
            plt.colorbar(sc2, ax=axes[1], label="log(1 + WT volume)")
            axes[1].set_title(f"{mn} — t-SNE by tumor volume")
        else:
            axes[1].text(0.5, 0.5, "No volume data", ha="center", transform=axes[1].transAxes)

        plt.tight_layout()
        out = FIG_DIR / f"{mn}_tsne.png"
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  Saved: {out}")

    # Norm + PCA distribution plots
    for mn, emb_dict in models.items():
        if not emb_dict: continue
        keys = sorted(emb_dict.keys())
        X = np.stack([emb_dict[k] for k in keys])
        norms = np.linalg.norm(X, axis=1)
        X_norm = X / (norms[:, None] + 1e-8)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].hist(norms, bins=50, color="steelblue", alpha=0.8)
        axes[0].axvline(norms.mean(), color="red", ls="--", label=f"mean={norms.mean():.2f}")
        axes[0].set_title(f"{mn} — Embedding L2 norms"); axes[0].legend()

        pca = PCA(n_components=min(100, X_norm.shape[1])).fit(X_norm)
        axes[1].plot(np.cumsum(pca.explained_variance_ratio_), color="darkorange", lw=2)
        axes[1].axhline(0.95, color="gray", ls="--", alpha=0.5, label="95%")
        axes[1].set_title(f"{mn} — PCA cumulative variance"); axes[1].legend()

        svs = np.linalg.svd(X_norm[:min(500, len(X_norm))], compute_uv=False)
        axes[2].plot(svs[:100], color="green", lw=2)
        axes[2].set_title(f"{mn} — Singular value spectrum")

        plt.tight_layout()
        out = FIG_DIR / f"{mn}_distributions.png"
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  Saved: {out}")

print(f"\nAll outputs in: {OUTPUT_ROOT}")

In [ ]:
# M2 UNIFIED EVALUATION — COMPUTED SUMMARY
print("\n" + "="*60)
print("  PHASE M2 EMBEDDING EVALUATION — MU-Glioma-Post")
print("="*60)

for mn in models:
    r = results.get(mn, {})
    print(f"\nModel: {mn}")
    emb_vals = list(models[mn].values()) if models[mn] else []
    dim = len(emb_vals[0]) if emb_vals else 0
    n_scans = len(models[mn]) if models[mn] else 0
    n_pats = len(set(k.split("__")[0] for k in models[mn].keys())) if models[mn] else 0
    print(f"  Scans: {n_scans} | Patients: {n_pats} | Embedding dim: {dim}")

    print(f"\n  MORPHOLOGY:")
    for k in ["M1_spearman_rho", "M1_volume_R2_rf", "M2_logvol_R2_rf",
              "M4_necrosis_F1", "M6_patient_purity_pct"]:
        v = r.get(k)
        if v is not None: print(f"    {k:<35s} = {v:.3f}")

    print(f"\n  HETEROGENEITY:")
    for k in ["H1_rankme", "H2_diversity", "H3_responder_F1", "H4_norm_cv"]:
        v = r.get(k)
        if v is not None:
            fmt = ".1f" if "rankme" in k else ".3f"
            print(f"    {k:<35s} = {v:{fmt}}")

    print(f"\n  TEMPORAL:")
    for k in ["T1_spearman_wt", "T4_rano_auc", "T7_treatment_d", "T8_kendall_tau"]:
        v = r.get(k)
        if v is not None: print(f"    {k:<35s} = {v:.3f}")

    # Strengths & weaknesses
    THRESH2 = {
        "M1_spearman_rho": 0.55, "M4_necrosis_F1": 0.60, "H1_rankme": 30.0,
        "H2_diversity": 0.25, "T5_coherence": 0.70, "H3_responder_F1": 0.55,
        "M6_patient_purity_pct": 60.0, "T4_rano_auc": 0.65,
        "T7_treatment_d": 0.50, "T8_kendall_tau": 0.30
    }
    strengths, weaknesses = [], []
    for k, thresh in THRESH2.items():
        v = r.get(k)
        if v is None: continue
        if v >= thresh:
            strengths.append(f"{k} = {v:.3f} (>{thresh})")
        else:
            weaknesses.append(f"{k} = {v:.3f} (needs >{thresh})")

    print(f"\n  STRENGTHS ({len(strengths)}):")
    for s in strengths: print(f"    ✅ {s}")
    print(f"\n  LIMITATIONS ({len(weaknesses)}) → Phase M3 targets:")
    for w in weaknesses: print(f"    ❌ {w}")

# ── Model comparison table (if both models present) ──
if len(results) >= 2:
    print("\n" + "="*60)
    print("  CNN vs ViT COMPARISON")
    print("="*60)
    compare_keys = ["M1_spearman_rho", "M1_volume_R2_rf", "M2_logvol_R2_rf",
                    "M4_necrosis_F1", "M6_patient_purity_pct",
                    "H1_rankme", "H2_diversity", "H3_responder_F1",
                    "T1_spearman_wt", "T4_rano_auc", "T7_treatment_d", "T8_kendall_tau"]
    model_names = list(results.keys())
    header = f"  {'Metric':<35s}"
    for mn in model_names:
        header += f"  {mn:>12s}"
    header += "    Winner"
    print(header)
    print("  " + "-"*len(header))
    for k in compare_keys:
        row = f"  {k:<35s}"
        vals = []
        for mn in model_names:
            v = results[mn].get(k)
            if v is not None:
                row += f"  {v:>12.3f}"
                vals.append(v)
            else:
                row += f"  {'N/A':>12s}"
                vals.append(None)
        if all(v is not None for v in vals) and len(vals) == 2:
            winner = model_names[0] if vals[0] > vals[1] else model_names[1]
            row += f"    {winner}"
        print(row)

print("\n" + "="*60)
print("  PHASE M2 EVALUATION COMPLETE")
print("="*60)
print(f"  Results: {OUTPUT_ROOT / 'm2_eval_results.json'}")
print(f"  Figures: {FIG_DIR}")
print(f"  → Next: Phase M3 TaViT V3 training")